# Clusterização: Lidando com dados sem rótulo

Atividade prática do curso da Alura sobre clusterização e preparação de dados sem rótulos.

O objetivo é carregar uma base de marketing, tratar a variável categórica `sexo` com One-Hot Encoding e treinar um modelo de clusterização com KMeans.

## 1. Importação das bibliotecas


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import OneHotEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import joblib


## 2. Carregamento dos dados


In [ ]:
url = 'https://raw.githubusercontent.com/alura-cursos/Clusterizacao-dados-sem-rotulo/main/Dados/dados_mkt.csv'

df = pd.read_csv(url)
df.head()


## 3. Exploração inicial


In [ ]:
df.info()


In [ ]:
df['sexo'].unique()


A coluna `sexo` possui valores categóricos. Para utilizar essa coluna no modelo, precisamos transformá-la em dados numéricos.

Para isso, utilizaremos o **One-Hot Encoding**, criando uma coluna binária para cada categoria.

## 4. Aplicação do One-Hot Encoding


In [ ]:
encoder = OneHotEncoder(categories=[['F', 'M', 'NE']], sparse_output=False)

encoded_sexo = encoder.fit_transform(df[['sexo']])
encoded_df = pd.DataFrame(
    encoded_sexo,
    columns=encoder.get_feature_names_out(['sexo'])
)

encoded_df.head()


## 5. Preparação da base final


In [ ]:
dados = pd.concat([df, encoded_df], axis=1).drop('sexo', axis=1)

dados.head()


## 6. Salvando o encoder treinado


In [ ]:
joblib.dump(encoder, 'encoder.pkl')


Salvar o encoder é importante porque, em um cenário real, os próximos dados precisam passar pela mesma transformação utilizada no treinamento.

## 7. Treinamento do modelo KMeans


In [ ]:
mod_kmeans = KMeans(n_clusters=2, random_state=45, n_init='auto')
modelo = mod_kmeans.fit(dados)

modelo


## 8. Adicionando os clusters ao DataFrame


In [ ]:
df_resultado = df.copy()
df_resultado['cluster'] = modelo.labels_

df_resultado.head()


## 9. Conferindo a quantidade de registros por cluster


In [ ]:
df_resultado['cluster'].value_counts()


## 10. Avaliação inicial com Silhouette Score


In [ ]:
score = silhouette_score(dados, modelo.labels_)
print(f'Silhouette Score: {score:.4f}')


O Silhouette Score ajuda a ter uma noção inicial da separação entre os clusters. Quanto mais próximo de 1, melhor tende a ser a separação entre os grupos.

## 11. Salvando o modelo e os dados clusterizados


In [ ]:
joblib.dump(modelo, 'modelo_kmeans.pkl')
df_resultado.to_csv('dados_clusterizados.csv', index=False)


## Conclusão

Nesta atividade, foi possível aplicar um fluxo básico de aprendizado não supervisionado. Como a base não possuía rótulos, o modelo KMeans foi utilizado para encontrar agrupamentos nos dados.

Antes do treinamento, foi necessário transformar a coluna categórica `sexo` em colunas numéricas por meio do One-Hot Encoding. Esse processo permitiu que os dados fossem utilizados corretamente pelo algoritmo de clusterização.